# Tests des fonctions de calcul de la saturation

In [51]:
import json
from datetime import date, timedelta

import pandas as pd
from pandas import NamedAgg

#from saturation_image_quali_prod import (
from saturation_image_quali import (
    filter_sessions_duration,
    filter_statuses_sessions,
    get_sampled_state_poc,
    to_sampled_state_grp,
    to_state_grp_d,
    to_state_poc_d,
)

SAMPLES: int = 288  # 5 min
SATURE_H: int = 45  # minimum duration (min) of saturation to have a saturated hour
MAX_SESSION_DURATION_HOURS = 24

ID_POC: str = "id_pdc_itinerance"
ID_STATION: str = "id_station_itinerance"
ID_POOL: str = "id_pool"
SATURATION_RATIO = 0.1
OVERLOAD_RATIO = 0.2
MIN_POWER = 75

#day = date(2026,7, 5)
#day = date(2026,7, 14)
day = date(2026,8, 1)
date_file = f"{day.year}{day.month:02d}{day.day:02d}"

data_quali = "../data/"

In [60]:
def read_statics(day: date, min_power: float) -> pd.DataFrame:
    """Read static data for pocs and stations."""
    date_statics = f"{day.day:02d}-{day.month:02d}-{day.year}"
    e5_str = pd.read_csv(f"../data_DMR_e2_e3/e5_{date_statics}.csv")["extras"][0]
    statics = pd.DataFrame(json.loads(e5_str))
    statics["unite"] = statics["id_pdc_itinerance"].str[:5]
    return statics[statics["puissance_nominale"] >= min_power]

def read_statics_pools(day:date, min_power: float) -> pd.DataFrame:
    """Read static data for stations and pools."""
    e1_statics = pd.read_csv("./tests_DMR/aires_pdc_2026-07-25.csv")[[ID_POOL, ID_STATION]].drop_duplicates()
    return e1_statics

In [61]:
e1_statics = read_statics_pools(day, MIN_POWER)
e1_statics

,id_pool,id_station_itinerance
0,A000001,FRHPCPNF080371TIERSTOTEM
13,A000002,FRTSLP5670
14,A000002,FRIOYP13531046
29,A000003,FRIOYP13530804
51,A000004,FRFASP11568703
...,...,...
7112,A001871,NaN
7113,A001872,NaN
7114,A001873,NaN
7115,A001874,NaN


In [3]:
def get_state_poc_for_chunk(
    day: date,
    samples_per_day: int,
    statics_chunk: pd.DataFrame,
    sessions: pd.DataFrame,
    statuses: pd.DataFrame,
) -> pd.DataFrame:
    """Calculate sampled_state_poc for a chunk."""
    statuses_chunk, sessions_chunk = filter_statuses_sessions(
        sessions, statuses, statics_chunk
    )
    sampled_state_poc_chunk = get_sampled_state_poc(
        day,
        samples_per_day,
        sessions_chunk,
        statuses_chunk,
    )
    state_poc_d_chunk = to_state_poc_d(sampled_state_poc_chunk, samples_per_day)
    return (sampled_state_poc_chunk, state_poc_d_chunk)

def get_chunked_state_poc(statics, day, samples_per_day, chunk_size, sessions, statuses):
    chunks = [
        statics.iloc[i : i + chunk_size] for i in range(0, len(statics), chunk_size)
    ]
    futures = [
        get_state_poc_for_chunk(#).submit(
            day,
            samples_per_day,
            chunk,
            sessions,
            statuses,
        )  # type: ignore[call-overload]
        for chunk in chunks
    ]
    #wait(futures)

    sampled_state_poc = pd.concat(
        [future[0] for future in futures], ignore_index=True
    )
    state_poc_d = pd.concat([future[1] for future in futures], ignore_index=True)
    return (sampled_state_poc, state_poc_d)

def get_chunked_state_grp(statics, sampled_state_poc, chunk_size, id_grp, samples_per_day, saturation_ratio, overload_ratio, add_full_use, add_latency):

    codes, _ = pd.factorize(statics[id_grp])
    statics["chunk"] = codes // chunk_size
    chunks = statics.groupby("chunk")
    #print(len(chunks))

    futures = [
        to_state_grp_d(
            to_sampled_state_grp(
                sampled_state_poc[sampled_state_poc[ID_POC].isin(chunk[ID_POC])],#.sort_values(by=["id_pdc_itinerance", "periode"]).reset_index(drop=True),
                chunk,
                id_grp,
                saturation_ratio,
                overload_ratio,
                add_full_use,
                add_latency
            ),  # type: ignore[call-overload]
            id_grp, 
            samples_per_day
        )
        for _, chunk in chunks
    ]

    state_grp = pd.concat(
        [future for future in futures], ignore_index=True
    )
    return state_grp

## test local

In [4]:
sessions = pd.read_csv('../data_test/donnees_sessions_FRHPCPNF050462_05-07-2026.csv')[['start', 'end', 'id_pdc_itinerance']]
sessions['start'] = pd.to_datetime(sessions['start'])
sessions['end'] = pd.to_datetime(sessions['end'])
statuses = pd.DataFrame({ID_POC:[], "horodatage":[], "etat_pdc":[], "occupation_pdc":[]})
statuses['horodatage'] = pd.to_datetime(statuses['horodatage'], utc=True)
# statuses = pd.DataFrame()
statics = pd.DataFrame({ID_POC:['FRHPCENF050462001', 'FRHPCENF050462002', 'FRHPCENF050462004' ], ID_STATION:['FRHPCPNF050462']*3})

samples_per_day = 288
#sampled_state_poc = get_sampled_state_poc(day, samples_per_day, sessions, statuses)

In [5]:
#sampled_state_poc[sampled_state_poc[ID_POC] == 'FRHPCENF050462001']

In [6]:
#state_poc_d = to_state_poc_d(sampled_state_poc, samples_per_day)

In [7]:
#state_poc_d, e2_pdc

In [8]:
#sample_state_station = to_sampled_state_grp(sampled_state_poc, statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO)
#state_station_h = to_state_grp_h(sample_state_station, ID_STATION, SAMPLES, SATURE_H)
#state_station_d = to_state_grp_d(state_station_h, ID_STATION)

In [9]:
#state_station_d

In [10]:
#e3_station

## test global

In [11]:
sessions_s3 = pd.read_parquet(data_quali + "qualicharge-" + date_file + "/sessions/production.parquet", engine="pyarrow")
statuses_s3 = pd.read_parquet(data_quali + "qualicharge-" + date_file + "/statuses/production.parquet", engine="pyarrow")

In [12]:

samples_per_day = SAMPLES #72 #288

e2_str = pd.read_csv("../data_DMR_e2_e3/e2_e3_05-07-2026.csv")["extras"][0]
e2_pdc = pd.DataFrame(json.loads(e2_str))

e3_str = pd.read_csv("../data_DMR_e2_e3/e2_e3_05-07-2026.csv")["extras"][1]
e3_station = pd.DataFrame(json.loads(e3_str))

e5_statics = read_statics(day, MIN_POWER)

# FRHPCENF050462001
e2_pdc = e2_pdc[e2_pdc[ID_POC].str[:14] == 'FRHPCENF050462']
e3_station = e3_station[e3_station[ID_STATION] == 'FRHPCPNF050462']


In [13]:
#e5_statics[e5_statics[ID_STATION] == 'FRHPCPNF080266TOTEM']
#e5_statics

In [14]:
# samples_per_day = 288

min_duration = timedelta(minutes=24 * 60 / samples_per_day)
max_duration = timedelta(hours=MAX_SESSION_DURATION_HOURS)
sessions = filter_sessions_duration(sessions_s3, min_duration=min_duration, max_duration=max_duration)
sessions_poc = sessions.groupby(ID_POC).agg(
                sessions_nb=NamedAgg("energy", "count"),
                energy_cum=NamedAgg("energy", "sum")
            ).reset_index()


In [15]:
#e5_statics[[ID_POC, ID_STATION]]
sessions_stations = pd.merge(e5_statics[[ID_POC, ID_STATION]], sessions_poc, on=ID_POC, how='left').fillna(0)
info_sessions_stations = sessions_stations[[ID_STATION, 'sessions_nb', 'energy_cum']].groupby(ID_STATION).sum().reset_index()
info_sessions_stations


,id_station_itinerance,sessions_nb,energy_cum
0,FR3R3P89882136,4,117.82
1,FRALDPFR00916,10,289.894
2,FRALDPFR00950,18,588.637
3,FRALLPGO000007,26,839.969
4,FRALLPGO000013,97,3168.213
...,...,...,...
6161,FRZUNP6023950095875504781,107,4121.733
6162,FRZUNP6927750076048540479,23,825.256
6163,FRZUNP7329346578064027187,223,9047.202
6164,FRZUNP8610050047391683219,23,742.349


### point de recharge

In [16]:
statuses_f, sessions_f = filter_statuses_sessions(sessions, statuses_s3, e5_statics)
sampled_state_poc_g = get_sampled_state_poc(day, samples_per_day, sessions_f, statuses_f)

In [17]:
#chunk_size = 200
#sampled_state_poc_chunk, state_poc_d_chunk = get_chunked_state_poc(e5_statics, day, samples_per_day, chunk_size, sessions, statuses_s3)

In [18]:
#sampled_sessions[sampled_sessions[ID_POC] == 'FRA79E12346905331']
#sampled_statuses[sampled_statuses[ID_POC] == 'FRA79E12346905331'][100:150]
#sampled_state_poc_g[sampled_state_poc_g['pseudo_occupe'] >0]
#sampled_state_poc_g[sampled_state_poc_g[ID_POC] == 'FRA79E12346905331'][0:100]
#sessions_s3[sessions_s3[ID_POC] == 'FRA79E12346905331']
#statuses_s3[statuses_s3[ID_POC] == 'FRA79E12346905331']
#sessions_s3[sessions_s3[ID_POC] == 'FRHPCENF050462002']
sampled_state_poc_g

,id_pdc_itinerance,periode,state,pseudo_libre,pseudo_occupe
0,FR3R3E10001456611,2026-08-01 00:00:00+00:00,libre,False,False
1,FR3R3E10001456611,2026-08-01 00:05:00+00:00,libre,False,False
2,FR3R3E10001456611,2026-08-01 00:10:00+00:00,libre,False,False
3,FR3R3E10001456611,2026-08-01 00:15:00+00:00,libre,False,False
4,FR3R3E10001456611,2026-08-01 00:20:00+00:00,libre,False,False
...,...,...,...,...,...
6725576,FRZUNEFR8801ER04,2026-08-01 23:35:00+00:00,libre,False,False
6725577,FRZUNEFR8801ER04,2026-08-01 23:40:00+00:00,libre,False,False
6725578,FRZUNEFR8801ER04,2026-08-01 23:45:00+00:00,libre,False,False
6725579,FRZUNEFR8801ER04,2026-08-01 23:50:00+00:00,libre,False,False


In [19]:
#sampled_state_poc_chunk

In [20]:
state_poc_d_g = to_state_poc_d(sampled_state_poc_g, samples_per_day)
state_poc_d_g

,id_pdc_itinerance,occupe,occupe_max,hors_service,libre,pseudo_libre,pseudo_occupe
0,FR3R3E10001456611,70.0,60.0,0.0,1370.0,70.0,0.0
1,FR3R3E10001456612,80.0,45.0,0.0,1360.0,80.0,0.0
2,FRALDE100541,145.0,55.0,0.0,1295.0,0.0,0.0
3,FRALDE100551,170.0,60.0,0.0,1270.0,0.0,15.0
4,FRALDE100552,90.0,45.0,0.0,1350.0,0.0,0.0
...,...,...,...,...,...,...,...
23346,FRZUNEFR8601ER06,220.0,55.0,15.0,1205.0,0.0,5.0
23347,FRZUNEFR8801ER01,175.0,60.0,0.0,1265.0,0.0,0.0
23348,FRZUNEFR8801ER02,140.0,55.0,0.0,1300.0,0.0,0.0
23349,FRZUNEFR8801ER03,65.0,35.0,0.0,1375.0,0.0,5.0


In [21]:
#state_poc_d_chunk.sort_values(by=["id_pdc_itinerance"])

In [22]:
#full_state_poc_d_g = add_sessions_info(state_poc_d_g, sessions, ID_POC)
full_state_poc_d_g = pd.merge(state_poc_d_g, sessions_poc, on=ID_POC, how='left').fillna(0)
len(full_state_poc_d_g)

23351

In [23]:
#state_poc_d_g[state_poc_d_g[ID_POC].str[:14] == 'FRHPCENF050462']
#state_poc_d_g[state_poc_d_g['pseudo_libre'] > 0]
#state_poc_d_g[state_poc_d_g['pseudo_occupe'] > 0]
full_state_poc_d_g[full_state_poc_d_g['sessions_nb'] > 30]

,id_pdc_itinerance,occupe,occupe_max,hors_service,libre,pseudo_libre,pseudo_occupe,sessions_nb,energy_cum
2973,FRELCE2VEE,935.0,60.0,0.0,505.0,10.0,30.0,33,1105.588
3058,FRELCE4BZP,1020.0,60.0,0.0,420.0,20.0,55.0,32,948.727
3177,FRELCE68ER,1055.0,60.0,0.0,385.0,15.0,40.0,35,1130.452
3607,FRELCECFS8,1135.0,60.0,0.0,305.0,15.0,80.0,43,1245.044
3628,FRELCECRWS,960.0,60.0,0.0,480.0,20.0,20.0,31,984.789
...,...,...,...,...,...,...,...,...,...
21983,FRTSLEM6OJ0R,870.0,60.0,0.0,570.0,10.0,55.0,32,1167.0212
21987,FRTSLEM6OKZG,910.0,60.0,0.0,530.0,25.0,125.0,33,1055.4193
21995,FRTSLEMALQHO,1015.0,60.0,0.0,425.0,5.0,50.0,31,1101.8298
22237,FRVIAE10001181312,680.0,60.0,0.0,760.0,0.0,50.0,32,1071.991


### station

In [24]:
sampled_state_station_g = to_sampled_state_grp(sampled_state_poc_g, e5_statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO)
print(len(sampled_state_station_g))

1579104


In [25]:
sampled_state_station_g_pu = to_sampled_state_grp(sampled_state_poc_g, e5_statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO, add_full_use=True, add_latency=True)
print(len(sampled_state_station_g_pu))

1579104


In [26]:
#sampled_state_station_g_pu.sort_values(by=[ID_STATION, "periode"])[100:150]

In [27]:
chunk_size = 200
state_station_chunk = get_chunked_state_grp(e5_statics, sampled_state_poc_g, chunk_size, ID_STATION, SAMPLES, SATURATION_RATIO, OVERLOAD_RATIO, add_full_use=True, add_latency=True)


In [28]:
state_station_chunk.sort_values(by=[ID_STATION])

,id_station_itinerance,nb_pdc,hs,inactif,pu_cum,sature_cum,surcharge,actif,sature_max,pu_max,pu_len
2869,FR3R3P89882136,2,0.0,1290.0,0.0,0.0,0.0,150.0,0.0,0.0,0.0
1992,FRALDPFR00916,8,0.0,1220.0,0.0,0.0,0.0,220.0,0.0,0.0,0.0
3219,FRALDPFR00950,8,0.0,995.0,0.0,0.0,5.0,440.0,0.0,0.0,0.0
917,FRALLPGO000007,6,0.0,830.0,340.0,205.0,170.0,235.0,60.0,60.0,230.0
3922,FRALLPGO000013,10,0.0,395.0,0.0,0.0,40.0,1005.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
3043,FRZUNP6023950095875504781,8,0.0,570.0,120.0,25.0,60.0,785.0,15.0,55.0,55.0
1820,FRZUNP6927750076048540479,8,0.0,1090.0,0.0,0.0,0.0,350.0,0.0,0.0,0.0
366,FRZUNP7329346578064027187,25,0.0,175.0,0.0,0.0,0.0,1265.0,0.0,0.0,0.0
367,FRZUNP8610050047391683219,6,0.0,1000.0,235.0,20.0,115.0,305.0,20.0,60.0,150.0


In [29]:
state_station_d_g_pu = to_state_grp_d(sampled_state_station_g_pu, ID_STATION, SAMPLES)
full_state_station_d_g_pu = pd.merge(state_station_d_g_pu, info_sessions_stations, on=ID_STATION, how='left').fillna(0)
#full_state_station_d_g_pu
state_station_d_g_pu

,id_station_itinerance,nb_pdc,hs,inactif,pu_cum,sature_cum,surcharge,actif,sature_max,pu_max,pu_len
0,FR3R3P89882136,2,0.0,1290.0,0.0,0.0,0.0,150.0,0.0,0.0,0.0
1,FRALDPFR00916,8,0.0,1220.0,0.0,0.0,0.0,220.0,0.0,0.0,0.0
2,FRALDPFR00950,8,0.0,995.0,0.0,0.0,5.0,440.0,0.0,0.0,0.0
3,FRALLPGO000007,6,0.0,830.0,340.0,205.0,170.0,235.0,60.0,60.0,230.0
4,FRALLPGO000013,10,0.0,395.0,0.0,0.0,40.0,1005.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
5478,FRZUNP6023950095875504781,8,0.0,570.0,120.0,25.0,60.0,785.0,15.0,55.0,55.0
5479,FRZUNP6927750076048540479,8,0.0,1090.0,0.0,0.0,0.0,350.0,0.0,0.0,0.0
5480,FRZUNP7329346578064027187,25,0.0,175.0,0.0,0.0,0.0,1265.0,0.0,0.0,0.0
5481,FRZUNP8610050047391683219,6,0.0,1000.0,235.0,20.0,115.0,305.0,20.0,60.0,150.0


In [30]:
sampled_state_station_g_pu[sampled_state_station_g_pu[ID_STATION] == 'FRALLPGO000007']

,id_station_itinerance,periode,occupe,hors_service,libre,pleine_utilisation,pseudo_libre,pseudo_occupe,nb_pdc,hs,inactif,pu,sature,surcharge,actif,state
864,FRALLPGO000007,2026-08-01 00:00:00+00:00,0,2,4,0,0,0,6,False,True,False,False,False,False,2
865,FRALLPGO000007,2026-08-01 00:05:00+00:00,0,2,4,0,0,0,6,False,True,False,False,False,False,2
866,FRALLPGO000007,2026-08-01 00:10:00+00:00,0,2,4,0,0,0,6,False,True,False,False,False,False,2
867,FRALLPGO000007,2026-08-01 00:15:00+00:00,0,2,4,0,0,0,6,False,True,False,False,False,False,2
868,FRALLPGO000007,2026-08-01 00:20:00+00:00,0,2,4,0,0,0,6,False,True,False,False,False,False,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1147,FRALLPGO000007,2026-08-01 23:35:00+00:00,0,2,4,0,0,0,6,False,True,False,False,False,False,2
1148,FRALLPGO000007,2026-08-01 23:40:00+00:00,0,2,4,0,0,0,6,False,True,False,False,False,False,2
1149,FRALLPGO000007,2026-08-01 23:45:00+00:00,0,2,4,0,0,0,6,False,True,False,False,False,False,2
1150,FRALLPGO000007,2026-08-01 23:50:00+00:00,0,2,4,0,0,0,6,False,True,False,False,False,False,2


In [31]:
sampled_state_station_g_pu[1147:1157]

,id_station_itinerance,periode,occupe,hors_service,libre,pleine_utilisation,pseudo_libre,pseudo_occupe,nb_pdc,hs,inactif,pu,sature,surcharge,actif,state
1147,FRALLPGO000007,2026-08-01 23:35:00+00:00,0,2,4,0,0,0,6,False,True,False,False,False,False,2
1148,FRALLPGO000007,2026-08-01 23:40:00+00:00,0,2,4,0,0,0,6,False,True,False,False,False,False,2
1149,FRALLPGO000007,2026-08-01 23:45:00+00:00,0,2,4,0,0,0,6,False,True,False,False,False,False,2
1150,FRALLPGO000007,2026-08-01 23:50:00+00:00,0,2,4,0,0,0,6,False,True,False,False,False,False,2
1151,FRALLPGO000007,2026-08-01 23:55:00+00:00,0,2,4,0,0,0,6,False,True,False,False,False,False,2
1152,FRALLPGO000013,2026-08-01 00:00:00+00:00,0,0,10,0,0,0,10,False,True,False,False,False,False,2
1153,FRALLPGO000013,2026-08-01 00:05:00+00:00,0,0,10,0,0,0,10,False,True,False,False,False,False,2
1154,FRALLPGO000013,2026-08-01 00:10:00+00:00,0,0,10,0,0,0,10,False,True,False,False,False,False,2
1155,FRALLPGO000013,2026-08-01 00:15:00+00:00,0,0,10,0,0,0,10,False,True,False,False,False,False,2
1156,FRALLPGO000013,2026-08-01 00:20:00+00:00,0,0,10,0,0,0,10,False,True,False,False,False,False,2


In [32]:

test = full_state_station_d_g_pu[(full_state_station_d_g_pu['sature_cum'] > 0) & (full_state_station_d_g_pu['nb_pdc'] > 2)]
test.to_csv('test.csv')
#test.to_excel('test.xlsx')

In [33]:
#e5_statics[e5_statics[ID_STATION] == 'FRPD1PBLDVDR']

In [34]:
#sessions_s3[sessions_s3[ID_POC] == 'FRPD1EBLDVDRKPC200015']

In [35]:
#statuses_s3[statuses_s3[ID_POC] == 'FRPD1EBLDVDRKPC200012']

In [36]:
sampled_state_station_g = to_sampled_state_grp(sampled_state_poc_g, e5_statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO)
print(len(sampled_state_station_g))

1579104


In [ ]:
def filter_sampled_state_grp(
    sampled_state_poc: pd.DataFrame, statics: pd.DataFrame
) -> pd.DataFrame:
    """Filter statuses and sessions with statics data."""
    filtered = sampled_state_poc[sampled_state_poc[ID_POC].isin(statics[ID_POC])].copy()
    return filtered

chunk_size = 200
codes, _ = pd.factorize(e5_statics[ID_STATION])
e5_statics["chunk"] = codes // chunk_size
chunks = e5_statics.groupby("chunk")
print(len(chunks))

futures = [
    to_sampled_state_grp(
        sampled_state_poc_g[sampled_state_poc_g[ID_POC].isin(chunk[ID_POC])],
        chunk,
        ID_STATION,
        SATURATION_RATIO,
        OVERLOAD_RATIO,
    )  # type: ignore[call-overload]
    for _, chunk in chunks
]

sampled_state_station = pd.concat(
    [future for future in futures], ignore_index=True
)
print(len(sampled_state_station))

31
1579104


In [38]:
state_station_d_g = to_state_grp_d(sampled_state_station_g, ID_STATION, SAMPLES)

In [39]:
#sampled_state_station_g
#sampled_state_station_g[sampled_state_station_g[ID_STATION] == 'FRHPCPNF080266TOTEM']
#state_station_h_g
#state_station_d_g
#state_station_d_g[state_station_d_g[ID_STATION] == 'FRHPCPNF080266TOTEM']
#state_station_d_g[state_station_d_g["sature_cum"] >= 120]

### Parc

In [80]:
e1_count = e1_statics.groupby(ID_POOL).agg(
    nb_stations=NamedAgg(column=ID_STATION, aggfunc="count")
)
single_station_pools = e1_count[e1_count["nb_stations"] == 1].index.tolist()
multi_station_pools = e1_count[e1_count["nb_stations"] > 1].index.tolist()
station_less_pools = e1_count[e1_count["nb_stations"] == 0].index.tolist()
print(len(single_station_pools), len(multi_station_pools), len(station_less_pools), len(e1_count))

pools_state_grp = e1_statics[e1_statics[ID_POOL].isin(single_station_pools)].merge(
    state_station_d_g, on=ID_STATION, how="left")
del pools_state_grp[ID_STATION]


pools_state_grp = pd.concat([pools_state_grp, pd.DataFrame(station_less_pools, columns=[ID_POOL])], ignore_index=True).fillna(0)

pools_state_grp



443 48 1384 1875


,id_pool,nb_pdc,hs,inactif,pu_cum,sature_cum,surcharge,actif,sature_max,pu_max,pu_len
0,A000001,11.0,0.0,660.0,0.0,0.0,0.0,780.0,0.0,0.0,0.0
1,A000003,16.0,0.0,385.0,0.0,10.0,30.0,1015.0,5.0,0.0,0.0
2,A000004,8.0,0.0,320.0,0.0,90.0,155.0,875.0,35.0,0.0,0.0
3,A000005,8.0,0.0,585.0,0.0,25.0,75.0,755.0,15.0,0.0,0.0
4,A000007,12.0,0.0,470.0,0.0,0.0,0.0,970.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
1822,A001871,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1823,A001872,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1824,A001873,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1825,A001874,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [86]:
pools_pdcs = e1_statics[e1_statics[ID_POOL].isin(multi_station_pools)].merge(e5_statics, on=ID_STATION, how="left")[[ID_POOL, ID_POC]]
#pools = read_statics_pools(day, MIN_POWER)
#pools_pdcs

In [83]:
sampled_state_pool = to_sampled_state_grp(sampled_state_poc_g, pools_pdcs, ID_POOL, SATURATION_RATIO, OVERLOAD_RATIO, add_full_use=True, add_latency=True)
print(len(sampled_state_pool))
sampled_state_pool

13824


,id_pool,periode,occupe,hors_service,libre,pleine_utilisation,pseudo_libre,pseudo_occupe,nb_pdc,hs,inactif,pu,sature,surcharge,actif,state
0,A000002,2026-08-01 00:00:00+00:00,0,0,16,0,0,0,16,False,True,False,False,False,False,2
1,A000002,2026-08-01 00:05:00+00:00,0,0,16,0,0,0,16,False,True,False,False,False,False,2
2,A000002,2026-08-01 00:10:00+00:00,0,0,16,0,0,0,16,False,True,False,False,False,False,2
3,A000002,2026-08-01 00:15:00+00:00,0,0,16,0,0,0,16,False,True,False,False,False,False,2
4,A000002,2026-08-01 00:20:00+00:00,0,0,16,0,0,0,16,False,True,False,False,False,False,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13819,A001698,2026-08-01 23:35:00+00:00,0,2,10,0,0,0,12,False,True,False,False,False,False,2
13820,A001698,2026-08-01 23:40:00+00:00,0,2,10,0,0,0,12,False,True,False,False,False,False,2
13821,A001698,2026-08-01 23:45:00+00:00,0,2,10,0,0,0,12,False,True,False,False,False,False,2
13822,A001698,2026-08-01 23:50:00+00:00,0,2,10,0,0,0,12,False,True,False,False,False,False,2


In [84]:
sessions_pools = pd.merge(pools_pdcs, sessions_poc, on=ID_POC, how='left').fillna(0)
info_sessions_pools = sessions_pools[[ID_POOL, 'sessions_nb', 'energy_cum']].groupby(ID_POOL).sum().reset_index()

In [85]:
state_pool = to_state_grp_d(sampled_state_pool, ID_POOL, SAMPLES)
full_state_pool = pd.merge(state_pool, info_sessions_pools, on=ID_POOL, how='left').fillna(0)
full_state_pool

,id_pool,nb_pdc,hs,inactif,pu_cum,sature_cum,surcharge,actif,sature_max,pu_max,pu_len,sessions_nb,energy_cum
0,A000002,16,0.0,170.0,10.0,0.0,40.0,1230.0,0.0,10.0,10.0,344,10077.15
1,A000010,19,0.0,310.0,0.0,0.0,20.0,1110.0,0.0,0.0,0.0,175,5981.057
2,A000032,2,0.0,415.0,715.0,545.0,0.0,480.0,55.0,60.0,145.0,66,2375.556
3,A000053,15,0.0,75.0,50.0,20.0,50.0,1295.0,10.0,30.0,30.0,335,11747.532
4,A000093,8,0.0,45.0,570.0,155.0,310.0,930.0,30.0,60.0,220.0,242,9424.88
5,A000173,20,0.0,50.0,0.0,0.0,10.0,1380.0,0.0,0.0,0.0,326,13844.532
6,A000198,20,0.0,310.0,0.0,0.0,0.0,1130.0,0.0,0.0,0.0,124,4792.096
7,A000221,16,0.0,175.0,50.0,5.0,55.0,1205.0,5.0,50.0,50.0,172,7054.88
8,A000322,16,0.0,665.0,0.0,0.0,0.0,775.0,0.0,0.0,0.0,72,2669.629
9,A000329,10,0.0,95.0,5.0,0.0,10.0,1335.0,0.0,5.0,5.0,193,6922.388


In [87]:
pools_state_grp = pd.concat([pools_state_grp, full_state_pool], ignore_index=True).fillna(0)


In [88]:
pools_state_grp

,id_pool,nb_pdc,hs,inactif,pu_cum,sature_cum,surcharge,actif,sature_max,pu_max,pu_len,sessions_nb,energy_cum
0,A000001,11.0,0.0,660.0,0.0,0.0,0.0,780.0,0.0,0.0,0.0,0,0.0
1,A000003,16.0,0.0,385.0,0.0,10.0,30.0,1015.0,5.0,0.0,0.0,0,0.0
2,A000004,8.0,0.0,320.0,0.0,90.0,155.0,875.0,35.0,0.0,0.0,0,0.0
3,A000005,8.0,0.0,585.0,0.0,25.0,75.0,755.0,15.0,0.0,0.0,0,0.0
4,A000007,12.0,0.0,470.0,0.0,0.0,0.0,970.0,0.0,0.0,0.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1870,A001268,14.0,0.0,30.0,20.0,15.0,35.0,1360.0,10.0,20.0,20.0,303,10197.331
1871,A001304,6.0,0.0,935.0,870.0,90.0,130.0,285.0,50.0,60.0,495.0,36,1478.178
1872,A001622,6.0,0.0,500.0,115.0,45.0,70.0,825.0,35.0,55.0,45.0,72,2542.898
1873,A001623,38.0,0.0,5.0,50.0,30.0,85.0,1320.0,25.0,50.0,50.0,728,29979.5264


### divers

In [44]:
static = pd.DataFrame({'station' : ['s1']*3+['s2']*5+['s3']*3+['s4']*4+['s5']*5,
                      'pdc': ['p'+str(i) for i in range(20)]})
#static

In [45]:
chunk = 2
station_to_chunk = {
    station: i // chunk
    for i, station in enumerate(static["station"].drop_duplicates())
}

static["chunk"] = static["station"].map(station_to_chunk)
#for chunk_id, chunk in static.groupby("chunk", sort=True):
#    print(chunk_id)
#    print(chunk)
#static

In [46]:
chunk_size = 3
'''station_to_chunk = {
    station: i // chunk_size
    for i, station in enumerate(static["station"].drop_duplicates())
}

static["chunk"] = static["station"].map(station_to_chunk)
chunks = static.groupby("chunk", sort=True)
'''
codes, _ = pd.factorize(static["station"])
static["chunk"] = codes//chunk_size
chunks = static.groupby("chunk")
#for chunk_id, chunk in chunks:
    #print(chunk_id)
    #print(chunk)
futures = [
    chunk for _ , chunk in chunks
    #to_sampled_state_grp(sampled_state_poc, statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO) for _ , chunk in chunks
] 
static_new = pd.concat(
        [future for future in futures], ignore_index=True
    )
#static_new

In [47]:
occupe_hs = pd.Series([False, True, False, True, False, False, True, True, False, False])
id_pdc = pd.Series(['p1', 'p1', 'p1', 'p1', 'p2', 'p2', 'p2', 'p2', 'p3', 'p3'])
tst = (occupe_hs | occupe_hs.shift(fill_value=False)) & (id_pdc == id_pdc.shift(fill_value=""))
tst

0    False
1     True
2     True
3     True
4    False
5    False
6     True
7     True
8    False
9    False
dtype: bool

In [48]:
occupe_hs.shift(fill_value=False)

0    False
1    False
2     True
3    False
4     True
5    False
6    False
7     True
8     True
9    False
dtype: bool

In [49]:
ids = id_pdc == id_pdc.shift(fill_value="")
ids

0    False
1     True
2     True
3     True
4    False
5     True
6     True
7     True
8    False
9     True
dtype: bool